<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 14: RAG Sistemi Giriş

**YAPAY ZEKA MÜHENDİSLİĞİ** · Modül 14 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta14/hafta14_rag_giris.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta14/hafta14_rag_giris.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>&nbsp;
<a href="https://raw.githubusercontent.com/DrMuratAltun/VB-YZ-90/main/web/public/sunumlar/hafta14_generative_ai.pdf"><img src="https://img.shields.io/badge/PDF%20Sunum-EC1C24?style=flat&logo=adobeacrobatreader&logoColor=white" alt="PDF Sunum"/></a>&nbsp;
<a href="https://drmurataltun.github.io/VB-YZ-90/hafta/14/"><img src="https://img.shields.io/badge/Web%20Sitesi-2B7A78?style=flat&logo=googlechrome&logoColor=white" alt="Web Sitesi"/></a>

</div>

---

**Eğitmen:** Dr. Murat Altun · [yapayzekaokulum.com](https://yapayzekaokulum.com) · [GitHub](https://github.com/DrMuratAltun)

**Program:** ECS Veri Bilimi ve Yapay Zeka Uzmanlığı · 90 Saat · 15 Hafta
---

> **Bu defterde neler öğreneceksiniz?**
>
> - RAG: Retrieval-Augmented Generation
> - ChromaDB ile vektör veritabanı
> - Doküman embedding ve arama

# Hafta 14 - RAG (Retrieval Augmented Generation) Giriş

Bu defterde, büyük dil modellerinin bilgi tabanını genişleten RAG sisteminin temellerini öğreneceğiz.

## Öğrenme Hedefleri
- RAG kavramını anlama
- Metin parçalama (text chunking)
- Embedding (gömme vektörleri) oluşturma
- ChromaDB ile vektör veritabanı kullanımı
- Benzerlik araması (similarity search)
- RAG pipeline: Soru → Arama → Bağlam → Yanıt
- RAG vs RAG'sız karşılaştırma

## RAG Nedir?

**RAG (Retrieval Augmented Generation)**, büyük dil modellerinin kendi eğitim verisinde olmayan bilgilere erişmesini sağlayan bir tekniktir.

### RAG Pipeline (İşlem Hattı)

```
┌──────────────┐     ┌──────────────────┐     ┌────────────────┐
│  1. BELGELER  │ ──→ │  2. PARÇALAMA    │ ──→ │  3. EMBEDDING  │
│  (Dökümanlar) │     │  (Chunking)      │     │  (Vektörler)   │
└──────────────┘     └──────────────────┘     └───────┬────────┘
                                                       │
                                                       ▼
                                              ┌────────────────┐
                                              │ 4. VEKTÖR DB   │
                                              │ (ChromaDB)     │
                                              └───────┬────────┘
                                                       │
┌──────────────┐     ┌──────────────────┐              │
│  7. YANIT    │ ←── │  6. LLM (Gemini) │ ←── ┌───────┴────────┐
│  (Cevap)     │     │  Bağlam + Soru   │     │ 5. BENZERLİK   │
└──────────────┘     └──────────────────┘     │    ARAMASI      │
                                              └────────────────┘
                              ▲
                     ┌───────┴────────┐
                     │ KULLANICI SORUSU│
                     └────────────────┘
```

### Neden RAG?
- LLM'ler eğitim verisiyle sınırlıdır (güncel bilgi yok)
- Şirket içi/özel belgeler LLM'de yoktur
- Halüsinasyon (uydurma bilgi) riskini azaltır
- Kaynağa dayalı, doğrulanabilir yanıtlar sağlar

In [ ]:
!pip install -q google-generativeai chromadb

### Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `IPython` | Yardımcı kütüphane |
| `chromadb` | Yardımcı kütüphane |
| `google` | Google Gemini AI API |


In [ ]:
import google.generativeai as genai
import chromadb
from IPython.display import Markdown, display

API_KEY = "YOUR_API_KEY"
genai.configure(api_key=API_KEY)

model = genai.GenerativeModel('gemini-2.5-flash')
print("Gemini ve ChromaDB hazır!")

## 1. Örnek Belge Koleksiyonu Oluşturma

Türk bilim tarihinden bir bilgi bankası oluşturalım. Bu paragraflar modelin eğitim verisinde **bu kadar detaylı** bulunmayan özel bilgilerdir.

In [ ]:
# Örnek belgeler: Türk bilim insanları ve katkıları
belgeler = [
    {
        "id": "doc1",
        "metin": """Cahit Arf (1910-1997), Türk matematik dünyasının en önemli isimlerindendir. 
        Arf Değişmezi (Arf Invariant) adıyla bilinen buluşu, cebirsel topoloji ve kuadratik formlar 
        teorisinde çığır açmıştır. Bu değişmez, düğüm teorisi ve manifold sınıflandırmasında 
        kritik bir araçtır. Arf, İstanbul Üniversitesi ve ODTÜ'de görev yapmıştır. 10 TL 
        banknotunun arkasında onun resmi ve Arf Değişmezi formülü yer almaktadır.""",
        "kaynak": "Türk Bilim Tarihi Ansiklopedisi"
    },
    {
        "id": "doc2",
        "metin": """Aziz Sancar (1946-), 2015 yılında Nobel Kimya Ödülü'nü kazanan Türk bilim 
        insanıdır. DNA onarım mekanizmalarını keşfetmesiyle tanınır. Nükleotid eksizyon onarımı 
        (NER) üzerine yaptığı çalışmalar, kanser tedavisinde yeni yollar açmıştır. Mardin Savur'da 
        doğan Sancar, İstanbul Tıp Fakültesi'nden mezun olmuş ve kariyerinin büyük bölümünü 
        ABD'de University of North Carolina'da sürdürmüştür. Kronobiyoloji alanında da 
        öncü çalışmaları bulunmaktadır.""",
        "kaynak": "Nobel Ödüllü Türk Bilim İnsanları"
    },
    {
        "id": "doc3",
        "metin": """Feza Gürsey (1921-1992), teorik fizik alanında dünya çapında tanınan bir 
        Türk fizikçidir. Parçacık fiziği ve grup teorisi alanlarındaki çalışmalarıyla bilinir. 
        Gürsey, kuark modelinin geliştirilmesinde önemli katkılarda bulunmuştur. SU(6) simetri 
        grubunun parçacık fiziğine uygulanması onun en önemli çalışmasıdır. Yale Üniversitesi'nde 
        uzun yıllar profesörlük yapmıştır. Oppenheimer Ödülü ve Wigner Madalyası almıştır.""",
        "kaynak": "Türk Fizik Derneği Arşivi"
    },
    {
        "id": "doc4",
        "metin": """Oktay Sinanoğlu (1935-2015), kuantum kimyası alanında çığır açan çalışmalar 
        yapmış bir Türk kimyagerdir. "Çok Elektronlu Atom Teorisi" ile tanınır. 29 yaşında Yale 
        Üniversitesi'nin en genç profesörü olmuştur. Moleküler orbital hesaplamalarında yeni 
        yöntemler geliştirmiştir. Ayrıca Türkçenin bilim dili olarak kullanılması için büyük 
        çaba harcamış ve "Bye Bye Türkçe" kitabıyla dil bilincini artırmaya çalışmıştır.""",
        "kaynak": "Türk Kimya Derneği"
    },
    {
        "id": "doc5",
        "metin": """İlhan Aksay, Princeton Üniversitesi'nde profesör olan Türk malzeme bilimcidir. 
        Grafen teknolojisi üzerine öncü çalışmalarıyla tanınır. Grafenin endüstriyel üretimine 
        yönelik yeni yöntemler geliştirmiştir. Nanoteknoloji ve seramik malzemeler alanında 
        200'den fazla bilimsel makalesi bulunmaktadır. Çalışmaları enerji depolama, biyomedikal 
        malzemeler ve ileri kompozitler gibi alanlara katkı sağlamaktadır.""",
        "kaynak": "Princeton Üniversitesi Profilleri"
    },
    {
        "id": "doc6",
        "metin": """Gazi Yaşargil (1925-), mikronöroşirürjinin (beyin cerrahisi) babası olarak 
        kabul edilir. İsviçre'de geliştirdiği mikrocerrahi teknikleri, dünya genelinde beyin 
        ameliyatlarını devrimsel şekilde değiştirmiştir. 1999'da "Yüzyılın Nöroşirürjeni" 
        seçilmiştir. Beyin damar hastalıkları, tümör cerrahisi ve epilepsi cerrahisinde 
        yenilikçi yaklaşımlar geliştirmiştir. Zürih Üniversitesi'nde uzun yıllar görev yapmıştır.""",
        "kaynak": "Dünya Nöroşirürji Tarihi"
    },
    {
        "id": "doc7",
        "metin": """Türkiye'de yapay zeka çalışmaları 1980'lerden itibaren üniversitelerde 
        başlamıştır. ODTÜ, Bilkent, Boğaziçi ve İTÜ öncü kurumlardır. 2021'de kurulan 
        Ulusal Yapay Zeka Stratejisi (UYZS) ile Türkiye, 2025 hedefleri belirlemiştir. 
        Strateji; eğitim, sağlık, tarım, ulaşım ve güvenlik alanlarında AI kullanımını 
        hedeflemektedir. TÜBİTAK BİLGEM bünyesindeki yapay zeka laboratuvarları, 
        savunma ve siber güvenlik projeleri yürütmektedir.""",
        "kaynak": "UYZS 2021-2025 Raporu"
    },
    {
        "id": "doc8",
        "metin": """Nükleer fizik alanında Türkiye'nin önemli isimlerinden biri Erdal İnönü'dür 
        (1926-2007). İsmet İnönü'nün oğlu olan Erdal İnönü, teorik fizik profesörüdür. 
        Poincaré grubunun temsilleri üzerine çalışmaları, Wigner-İnönü daralması olarak 
        bilinir ve Lie cebirleri teorisinde temel bir kavramdır. Ankara Üniversitesi ve 
        ODTÜ'de akademik kariyerini sürdürmüş, ayrıca siyasete girerek Başbakan Yardımcısı 
        olmuştur.""",
        "kaynak": "Türk Fizikçiler Biyografi Sözlüğü"
    }
]

print(f"Toplam {len(belgeler)} belge hazırlandı.")
for b in belgeler:
    print(f"  - {b['id']}: {b['metin'][:60]}...")

## 2. Metin Parçalama (Text Chunking)

Uzun belgeleri daha küçük parçalara ayırmak, arama doğruluğunu artırır.

In [ ]:
def metin_parcala(metin, parca_boyutu=200, cakisma=50):
    """Metni belirli boyutta parçalara ayır.
    
    Args:
        metin: Parçalanacak metin
        parca_boyutu: Her parçanın karakter sayısı
        cakisma: Parçalar arası çakışma (overlap) karakter sayısı
    """
    parcalar = []
    baslangic = 0
    
    while baslangic < len(metin):
        bitis = baslangic + parca_boyutu
        parca = metin[baslangic:bitis].strip()
        if parca:
            parcalar.append(parca)
        baslangic += parca_boyutu - cakisma
    
    return parcalar

# Tüm belgeleri parçala
tum_parcalar = []
parca_meta = []  # Her parçanın hangi belgeye ait olduğu

for belge in belgeler:
    # Boşlukları temizle
    temiz_metin = " ".join(belge["metin"].split())
    parcalar = metin_parcala(temiz_metin, parca_boyutu=300, cakisma=50)
    
    for i, parca in enumerate(parcalar):
        parca_id = f"{belge['id']}_chunk{i}"
        tum_parcalar.append({
            "id": parca_id,
            "metin": parca,
            "kaynak": belge["kaynak"],
            "belge_id": belge["id"]
        })

print(f"Toplam {len(tum_parcalar)} parça oluşturuldu.")
print("\nİlk 3 parça:")
for p in tum_parcalar[:3]:
    print(f"  [{p['id']}]: {p['metin'][:80]}...")

## 3. Embedding (Gömme Vektörleri) Oluşturma

Embedding, metnin anlamını sayısal vektörlere dönüştürür. Anlamca benzer metinler, vektör uzayında birbirine yakın olur.

```
"Kedi yatağında uyuyor" → [0.12, -0.34, 0.56, ...] ─┐
                                                       ├── Yakın (benzer anlam)
"Kediciğim uykuya daldı" → [0.11, -0.32, 0.55, ...] ─┘

"Araba hızla gidiyor"   → [0.87, 0.21, -0.44, ...] ──── Uzak (farklı anlam)
```

In [ ]:
def embedding_olustur(metin):
    """Gemini ile metin embedding'i oluştur."""
    result = genai.embed_content(
        model="models/embedding-001",
        content=metin,
        task_type="retrieval_document"
    )
    return result['embedding']

def soru_embedding_olustur(soru):
    """Soru için embedding oluştur (farklı task_type)."""
    result = genai.embed_content(
        model="models/embedding-001",
        content=soru,
        task_type="retrieval_query"
    )
    return result['embedding']

# Test: Bir metnin embedding'ini görelim
test_embedding = embedding_olustur("Cahit Arf Türk matematikçidir.")
print(f"Embedding boyutu: {len(test_embedding)}")
print(f"İlk 10 değer: {test_embedding[:10]}")

## 4. ChromaDB ile Vektör Veritabanı

ChromaDB, embedding'leri depolayan ve benzerlik araması yapan bir vektör veritabanıdır.

In [ ]:
# ChromaDB istemcisi oluştur
chroma_client = chromadb.Client()

# Koleksiyon oluştur (varsa sil ve yeniden oluştur)
koleksiyon_adi = "turk_bilim_insanlari"
try:
    chroma_client.delete_collection(koleksiyon_adi)
except:
    pass

koleksiyon = chroma_client.create_collection(
    name=koleksiyon_adi,
    metadata={"hnsw:space": "cosine"}  # Kosinüs benzerliği kullan
)

print(f"Koleksiyon '{koleksiyon_adi}' oluşturuldu.")

### Tüm parçaları embedding'le ve ChromaDB'ye ekle

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Tüm parçaları embedding'le ve ChromaDB'ye ekle
import time

print("Embedding'ler oluşturuluyor ve veritabanına ekleniyor...")

ids = []
embeddings = []
documents = []
metadatas = []

for i, parca in enumerate(tum_parcalar):
    print(f"  [{i+1}/{len(tum_parcalar)}] {parca['id']}...", end="")
    
    emb = embedding_olustur(parca['metin'])
    
    ids.append(parca['id'])
    embeddings.append(emb)
    documents.append(parca['metin'])
    metadatas.append({
        'kaynak': parca['kaynak'],
        'belge_id': parca['belge_id']
    })
    
    print(" Tamam")
    time.sleep(0.5)  # API rate limit için bekleme

# Toplu ekleme
koleksiyon.add(
    ids=ids,
    embeddings=embeddings,
    documents=documents,
    metadatas=metadatas
)

print(f"\nToplam {koleksiyon.count()} parça veritabanına eklendi.")

## 5. Benzerlik Araması (Similarity Search)

Kullanıcının sorusunu embedding'e çevirip veritabanında en benzer parçaları bulalım.

In [ ]:
def benzer_parcalari_bul(soru, n_sonuc=3):
    """Soruya en benzer belge parçalarını bul."""
    soru_emb = soru_embedding_olustur(soru)
    
    sonuclar = koleksiyon.query(
        query_embeddings=[soru_emb],
        n_results=n_sonuc
    )
    
    return sonuclar

# Test sorgusu
soru = "Nobel ödülü alan Türk bilim insanı kimdir?"
sonuclar = benzer_parcalari_bul(soru)

print(f"Soru: {soru}")
print(f"\nBulunan {len(sonuclar['documents'][0])} ilgili parça:")
print("-" * 50)

for i, (doc, meta, dist) in enumerate(zip(
    sonuclar['documents'][0],
    sonuclar['metadatas'][0],
    sonuclar['distances'][0]
)):
    print(f"\n[{i+1}] Benzerlik: {1 - dist:.3f} | Kaynak: {meta['kaynak']}")
    print(f"    {doc[:150]}...")

## 6. RAG Pipeline: Bağlam + Soru → Gemini → Yanıt

Şimdi tüm parçaları birleştirerek tam bir RAG sistemi kuralım.

In [ ]:
def rag_soru_sor(soru, n_sonuc=3):
    """RAG pipeline: Soru sor, ilgili belgeleri bul, Gemini ile yanıtla."""
    
    # 1. Benzer belgeleri bul
    sonuclar = benzer_parcalari_bul(soru, n_sonuc=n_sonuc)
    
    # 2. Bağlam oluştur
    baglam_parcalari = []
    kaynaklar = []
    
    for doc, meta in zip(sonuclar['documents'][0], sonuclar['metadatas'][0]):
        baglam_parcalari.append(doc)
        kaynaklar.append(meta['kaynak'])
    
    baglam = "\n\n".join(baglam_parcalari)
    
    # 3. Gemini'ye bağlam + soru gönder
    rag_prompt = f"""Aşağıdaki bağlam bilgilerine dayanarak soruyu yanıtla.

KURALLAR:
- SADECE verilen bağlamdaki bilgileri kullan
- Bağlamda olmayan bilgiyi UYDURMA
- Eğer bağlamda yeterli bilgi yoksa "Bu bilgi veritabanımda bulunmamaktadır" de
- Yanıtın sonunda kaynak belirt
- Türkçe yanıt ver

BAĞLAM:
{baglam}

SORU: {soru}

YANIT:"""
    
    response = model.generate_content(rag_prompt)
    
    return {
        'yanit': response.text,
        'kaynaklar': list(set(kaynaklar)),
        'baglam': baglam
    }

# RAG ile soru sor
sonuc = rag_soru_sor("Aziz Sancar hangi alanda Nobel ödülü aldı?")

print("YANIT:")
print(sonuc['yanit'])
print(f"\nKaynaklar: {', '.join(sonuc['kaynaklar'])}")

### Daha fazla soru deneyelim

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Daha fazla soru deneyelim
sorular = [
    "10 TL'nin arkasında hangi bilim insanı var?",
    "Mikronöroşirürjinin babası kimdir?",
    "Türkiye'nin yapay zeka stratejisi ne zaman oluşturuldu?",
    "Feza Gürsey hangi üniversitede çalıştı?",
    "Grafen teknolojisi üzerine çalışan Türk bilim insanı kim?"
]

for soru in sorular:
    print(f"\n{'='*60}")
    print(f"SORU: {soru}")
    print(f"{'='*60}")
    sonuc = rag_soru_sor(soru)
    print(f"YANIT: {sonuc['yanit']}")
    print(f"Kaynaklar: {', '.join(sonuc['kaynaklar'])}")

## 7. Karşılaştırma: RAG ile vs RAG'sız

RAG'ın farkını görmek için aynı soruyu hem RAG ile hem de RAG'sız soralım.

In [ ]:
def karsilastir(soru):
    """Aynı soruyu RAG ile ve RAG'sız karşılaştır."""
    
    print(f"SORU: {soru}")
    print("=" * 60)
    
    # RAG'sız (doğrudan Gemini)
    print("\n--- RAG'SIZ (Doğrudan Gemini) ---")
    ragsiz_prompt = f"Kısaca yanıtla: {soru}"
    ragsiz_yanit = model.generate_content(ragsiz_prompt)
    print(ragsiz_yanit.text)
    
    # RAG ile
    print("\n--- RAG İLE (Veritabanı Destekli) ---")
    rag_sonuc = rag_soru_sor(soru)
    print(rag_sonuc['yanit'])
    print(f"\nKaynaklar: {', '.join(rag_sonuc['kaynaklar'])}")

# Karşılaştırma: Spesifik soru
karsilastir("Wigner-İnönü daralması nedir ve kim geliştirdi?")

### Veritabanında olmayan bir soruyla test

Aşağıdaki kod bloğunda bu işlemi gerçekleştiriyoruz.

In [ ]:
# Veritabanında olmayan bir soruyla test
karsilastir("Einstein'ın görelilik teorisi nedir?")

## Özet

### RAG Pipeline Adımları

| Adım | İşlem | Araç |
|------|-------|------|
| 1 | Belgeleri topla | Python |
| 2 | Parçalara ayır (chunking) | `metin_parcala()` |
| 3 | Embedding oluştur | `genai.embed_content()` |
| 4 | Vektör DB'ye kaydet | ChromaDB |
| 5 | Soru embedding'i oluştur | `genai.embed_content()` |
| 6 | Benzer parçaları bul | ChromaDB `.query()` |
| 7 | Bağlam + soru → LLM | Gemini `generate_content()` |

### RAG'ın Avantajları
- Güncel ve özel bilgilere erişim
- Halüsinasyon riski azalır
- Kaynak gösterimi ile güvenilirlik artar
- Model yeniden eğitimi gerekmez

### Alıştırma
1. Kendi konunuzla ilgili 10 paragraf hazırlayın (ders notları, Wikipedia makaleleri vb.)
2. Bunları RAG pipeline'ına ekleyin
3. 5 farklı soru sorun ve RAG ile RAG'sız karşılaştırma yapın

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://scholargent.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

&copy; 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>